# Complete Pairwise Tanimoto Fold-Distance Tables — Hi Tasks

This notebook computes fold-to-fold structural distances for the Hi datasets used in the OOD-holdout vs random-shuffle analysis: DRD2, HIV and Sol.

The goal is to test whether the Lo-Hi structural split is also reflected in the activity-relevant feature subspace.

For each dataset and each fold pair, we compute distances in:

- the full ECFP4 space;
- global activity top-k ECFP4 bits;
- OOD-selected activity top-k ECFP4 bits;
- random-shuffle-selected activity top-k ECFP4 bits;
- random top-k ECFP4 bits as a dimensionality-control baseline.

The main distance is the complete cross-fold pairwise Tanimoto distance:

\[
D(A,B)=\frac{1}{|A||B|}\sum_{x\in A}\sum_{y\in B} \left(1-T(x,y)\right)
\]

where \(T(x,y)\) is the Tanimoto similarity between two ECFP4 fingerprints.

Nearest-neighbour distances are saved only as a secondary diagnostic. Restricted-space results must be interpreted together with the valid molecule fraction, because at small k some molecules may have all-zero restricted fingerprints.

Outputs are saved to:

`results/results_fold_distance_tanimoto/hi/`

In [1]:
import json
import sys
import time
import warnings
import zlib
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

try:
    from scipy.stats import wasserstein_distance_nd

    _HAS_WASSERSTEIN = True
except Exception:
    _HAS_WASSERSTEIN = False


def find_project_root(start: Path | None = None) -> Path:
    if start is None:
        start = Path.cwd().resolve()
    current = start
    while current != current.parent:
        if all((current / d).exists() for d in ["data", "utils"]):
            return current
        current = current.parent
    raise RuntimeError("Could not find project root containing data/ and utils/.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.fingerprints import compute_fingerprints

print(f"Project root: {PROJECT_ROOT}")
print(f"scipy wasserstein_distance_nd available: {_HAS_WASSERSTEIN}")

Project root: /home/f.capria/drug-discovery-lohi
scipy wasserstein_distance_nd available: True


In [2]:
TASK = "hi"
DATASETS_MAIN = ["drd2", "hiv", "sol"]
DATASETS = DATASETS_MAIN

DATASET_LABELS = {"drd2": "DRD2", "hiv": "HIV", "sol": "Sol"}

SUBSET_FILES = {"F1": "test_3.csv", "F2": "test_2.csv", "F3": "test_1.csv"}
PAIRS = list(combinations(["F1", "F2", "F3"], 2))

PAIR_TO_OUTER_FOLD = {"F1_vs_F2": 1, "F1_vs_F3": 2, "F2_vs_F3": 3}

FP_TYPE = "ecfp4"
EXPECTED_ECFP4_BITS = 2048

K_VALUES = [10, 20, 50, 100, 150, 200, 250, 500]

# Distances are computed separately for each model family used in the OOD-vs-random
MODELS = ["DT", "LR", "SVM"]

PAIRWISE_CHUNK_SIZE = 512
N_RANDOM_BIT_REPEATS = 30

VERBOSE_PAIRWISE = False
PRINT_EVERY_RANDOM_REPEAT = 5

RUN_WASSERSTEIN = False
W_SUBSAMPLE = 400

RANDOM_STATE = 42

DATA_ROOT = PROJECT_ROOT / "data" / TASK
OOD_CROSS_DIR = (
    PROJECT_ROOT / "results" / "results_ood_vs_random_shuffle" / TASK / "cross_dataset"
)
OUT_ROOT = PROJECT_ROOT / "results" / "results_fold_distance_tanimoto" / TASK
FIG_ROOT = OUT_ROOT / "figures"
CACHE_ROOT = PROJECT_ROOT / "features" / "fold_distance_tanimoto" / TASK

for d in (OUT_ROOT, FIG_ROOT, CACHE_ROOT):
    d.mkdir(parents=True, exist_ok=True)

print(f"OUT_ROOT: {OUT_ROOT}")
print(f"FIG_ROOT: {FIG_ROOT}")
print(f"CACHE_ROOT: {CACHE_ROOT}")

OUT_ROOT: /home/f.capria/drug-discovery-lohi/results/results_fold_distance_tanimoto/hi
FIG_ROOT: /home/f.capria/drug-discovery-lohi/results/results_fold_distance_tanimoto/hi/figures
CACHE_ROOT: /home/f.capria/drug-discovery-lohi/features/fold_distance_tanimoto/hi


In [3]:
MODEL_NAME_MAP = {
    "Decision Tree": "DT",
    "Logistic Regression": "LR",
    "Linear SVM": "SVM",
    "dt": "DT",
    "decision_tree": "DT",
    "lr": "LR",
    "logistic_regression": "LR",
    "svm": "SVM",
    "svm_linear": "SVM",
    "DT": "DT",
    "LR": "LR",
    "SVM": "SVM",
}

FP_MAP = {
    "ECFP4": "ecfp4",
    "MACCS": "maccs",
    "RDKit desc": "rdkit_desc",
    "ecfp4": "ecfp4",
    "maccs": "maccs",
    "rdkit_desc": "rdkit_desc",
}

IMPORTANCE_COL_CANDIDATES = [
    "importance_value",
    "permutation_importance_mean",
    "abs_weight",
    "normalized_abs_importance",
    "tree_importance",
]

In [5]:
def stable_seed(*parts, base: int = RANDOM_STATE) -> int:
    """Deterministic, cross-platform seed from arbitrary tuple of parts."""
    s = "|".join(str(p) for p in parts) + f"|{base}"
    return zlib.crc32(s.encode("utf-8")) & 0xFFFFFFFF


def local_rng(*parts) -> np.random.Generator:
    return np.random.default_rng(stable_seed(*parts))


def print_section(title: str) -> None:
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def get_smiles_col(df: pd.DataFrame) -> str:
    for col in ["smiles", "SMILES", "canonical_smiles"]:
        if col in df.columns:
            return col
    raise ValueError(f"No SMILES column found. Columns: {df.columns.tolist()}")


def find_importance_col(df: pd.DataFrame) -> str:
    if "importance_value" in df.columns:
        return "importance_value"
    for c in IMPORTANCE_COL_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(
        f"No importance column found among {IMPORTANCE_COL_CANDIDATES}. "
        f"Columns: {df.columns.tolist()}"
    )


def normalize_model_name(x) -> str:
    s = str(x).strip()
    return MODEL_NAME_MAP.get(s, s)


def normalize_fingerprint_name(x) -> str:
    s = str(x).strip()
    return FP_MAP.get(s, s)


def protocol_match(x) -> str:
    s = str(x).strip().lower()
    if s in {"ood holdout", "ood_holdout", "holdout_ood", "ood"}:
        return "ood"
    if s in {"random shuffle", "random_shuffle", "random"}:
        return "random"
    if "ood" in s and "random" not in s:
        return "ood"
    if "random" in s:
        return "random"
    return s

In [4]:
def load_subset_fps(dataset: str) -> dict[str, tuple[np.ndarray, pd.DataFrame]]:
    """Read F1/F2/F3 SMILES, compute ECFP4 with cache, drop invalid SMILES."""
    print_section(f"Loading subsets: {dataset.upper()}-{TASK}")

    data_dir = DATA_ROOT / dataset
    cache_dir = CACHE_ROOT / dataset
    cache_dir.mkdir(parents=True, exist_ok=True)

    out = {}
    for subset, fname in SUBSET_FILES.items():
        df = pd.read_csv(data_dir / fname).copy()
        smi_col = get_smiles_col(df)
        smiles = df[smi_col].astype(str).tolist()
        raw_n = len(smiles)

        t0 = time.time()
        X, valid_mask = compute_fingerprints(
            smiles_list=smiles,
            fp_type=FP_TYPE,
            cache_path=str(cache_dir / f"{subset}_{FP_TYPE}.npz"),
        )
        elapsed = time.time() - t0

        valid_mask = np.asarray(valid_mask, dtype=bool)
        if len(valid_mask) != raw_n:
            raise ValueError(
                f"valid_mask length mismatch: {len(valid_mask)} vs {raw_n}"
            )
        df_valid = df.loc[valid_mask].reset_index(drop=True)

        X = np.asarray(X, dtype=np.uint8)
        n_valid = X.shape[0]
        if n_valid != len(df_valid):
            raise ValueError(
                f"X/df mismatch after valid_mask: {n_valid} vs {len(df_valid)}"
            )
        if X.shape[1] != EXPECTED_ECFP4_BITS:
            raise ValueError(
                f"Expected {EXPECTED_ECFP4_BITS} bits, got {X.shape[1]} for {subset}"
            )

        invalid_n = raw_n - n_valid
        print(
            f"  {subset} ({fname}): raw={raw_n}, valid={n_valid}, "
            f"invalid={invalid_n}, n_bits={X.shape[1]}, time={elapsed:.1f}s"
        )

        out[subset] = (X, df_valid)

    return out


print("Fingerprint loading function defined.")

Fingerprint loading function defined.


Distance computation choices

- Full ECFP4 is the baseline structural distance and the reference for everything else.
- Activity-restricted spaces (`activity_global`, `activity_ood`, `activity_random_shuffle`) test whether the shift lies in features that the activity model actually relies on.
- Dataset-detection space (`dataset_detection`, from List B same-search CV) tests the distance induced by features that explicitly distinguish fold pairs such as F1_vs_F2.
- Random top-k bits (`random_bits`) is the negative control: if a restricted space gives the same distance as random restriction of the same dimensionality, the apparent shift is a dimensionality artefact, not evidence of activity- or detection-relevance. The key comparison is restricted-space distance minus random-bit distance.
- Valid-molecule fractions are required because at small k some molecules may have all-zero restricted fingerprints; restricted-space distances must be interpreted in light of how many molecules actually contribute.

In [6]:
def nn_max_sim(X: np.ndarray, Y: np.ndarray, chunk: int = 512) -> np.ndarray:
    """For each row in X, return max Tanimoto similarity to any row in Y."""
    X = X.astype(np.float32, copy=False)
    Y = Y.astype(np.float32, copy=False)
    sx = X.sum(axis=1)
    sy = Y.sum(axis=1)
    n = X.shape[0]
    out = np.zeros(n, dtype=np.float32)
    for i in range(0, n, chunk):  # chunk for the dimensionality
        block = X[i : i + chunk]
        sb = sx[i : i + chunk]
        dots = block @ Y.T  # intersection
        denom = sb[:, None] + sy[None, :] - dots  # union
        with np.errstate(divide="ignore", invalid="ignore"):
            sim = np.where(denom > 0, dots / denom, 0.0)
        out[i : i + chunk] = sim.max(axis=1)
    return out


def pair_sim(
    X: np.ndarray, Y: np.ndarray, idx_x: np.ndarray, idx_y: np.ndarray
) -> np.ndarray:
    """Tanimoto similarity over a set of paired rows. Returns NaN where union is zero."""
    xa = X[idx_x].astype(np.float32, copy=False)
    yb = Y[idx_y].astype(np.float32, copy=False)
    dots = (xa * yb).sum(axis=1)
    sa = xa.sum(axis=1)
    sb = yb.sum(axis=1)
    denom = sa + sb - dots
    with np.errstate(divide="ignore", invalid="ignore"):
        sim = np.where(denom > 0, dots / denom, np.nan)
    return sim


def nn_distances(XA: np.ndarray, XB: np.ndarray) -> dict:
    """Symmetric nearest-neighbour Tanimoto distance, restricted-space aware."""
    sa = XA.sum(axis=1)
    sb = XB.sum(axis=1)
    valid_a = sa > 0
    valid_b = sb > 0
    n_total_a = int(XA.shape[0])
    n_total_b = int(XB.shape[0])

    XA_v = XA[valid_a]
    XB_v = XB[valid_b]

    if XA_v.shape[0] == 0 or XB_v.shape[0] == 0:
        return {
            "nn_sym_distance": np.nan,
            "nn_A_to_B_mean": np.nan,
            "nn_B_to_A_mean": np.nan,
            "nn_AB_array": np.array([], dtype=np.float32),
            "nn_BA_array": np.array([], dtype=np.float32),
            "n_valid_a": int(valid_a.sum()),
            "n_valid_b": int(valid_b.sum()),
            "n_total_a": n_total_a,
            "n_total_b": n_total_b,
        }

    # to maintain simmetry
    sim_AB = nn_max_sim(XA_v, XB_v)
    sim_BA = nn_max_sim(XB_v, XA_v)
    d_AB = 1.0 - sim_AB
    d_BA = 1.0 - sim_BA

    return {
        "nn_sym_distance": float(
            0.5 * (d_AB.mean() + d_BA.mean())
        ),  # mean for simmetry
        "nn_A_to_B_mean": float(d_AB.mean()),
        "nn_B_to_A_mean": float(d_BA.mean()),
        "nn_AB_array": d_AB,
        "nn_BA_array": d_BA,
        "n_valid_a": int(valid_a.sum()),
        "n_valid_b": int(valid_b.sum()),
        "n_total_a": n_total_a,
        "n_total_b": n_total_b,
    }


def complete_pairwise_distance(
    XA: np.ndarray,
    XB: np.ndarray,
    dataset: str,
    pair: str,
    space: str,
    chunk: int = PAIRWISE_CHUNK_SIZE,
) -> dict:
    """
    Complete cross-fold pairwise Tanimoto distance.

    Computes the mean Tanimoto distance over all pairs
    """
    XA = XA.astype(np.float32, copy=False)
    XB = XB.astype(np.float32, copy=False)

    sum_dist = 0.0
    n_valid = 0
    n_total = int(XA.shape[0] * XB.shape[0])
    if VERBOSE_PAIRWISE:
        print(
            f"      pairwise {dataset} | {pair} | {space}: "
            f"{XA.shape[0]} x {XB.shape[0]} = {n_total:,} pairs"
        )

    y_sum = XB.sum(axis=1)

    for start in range(0, XA.shape[0], chunk):
        xb = XA[start : start + chunk]
        x_sum = xb.sum(axis=1)

        inter = xb @ XB.T
        union = x_sum[:, None] + y_sum[None, :] - inter

        valid = union > 0

        sim = np.zeros_like(inter, dtype=np.float32)
        np.divide(inter, union, out=sim, where=valid)

        dist = 1.0 - sim

        sum_dist += float(dist[valid].sum())
        n_valid += int(valid.sum())

    mean_dist = sum_dist / n_valid if n_valid > 0 else np.nan

    if VERBOSE_PAIRWISE:
        print(
            f"        done: mean={mean_dist:.4f}, "
            f"valid_pairs={n_valid:,}/{n_total:,} "
            f"({n_valid / n_total:.3f})"
        )

    return {
        "pairwise_distance": mean_dist,
        "n_valid_pairs": n_valid,
        "n_total_pairs": n_total,
        "valid_pair_fraction": n_valid / n_total if n_total > 0 else np.nan,
        "pairwise_mode": "complete",
    }

In [7]:
def load_list_a() -> pd.DataFrame:
    """Load List A activity feature importance, normalised and ECFP4-only."""
    print_section("Loading List A")

    path = OOD_CROSS_DIR / "cross_dataset_feature_importance_all.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing List A file: {path}")

    df = pd.read_csv(path, low_memory=False).copy()
    print(f"Raw List A: {df.shape}")

    df["dataset"] = df["dataset"].astype(str).str.lower()
    df = df[df["dataset"].isin(DATASETS_MAIN)].copy()

    df["model"] = df["model"].map(lambda x: normalize_model_name(x)).astype(str)
    df["fingerprint"] = (
        df["fingerprint"].map(lambda x: normalize_fingerprint_name(x)).astype(str)
    )
    df = df[df["fingerprint"] == "ecfp4"].copy()

    imp_col = find_importance_col(df)
    print(f"Using importance column: {imp_col}")

    df["importance_value_numeric"] = df[imp_col].fillna(0).astype(float)

    df["abs_importance"] = np.where(
        df["model"].eq("DT"),
        df["importance_value_numeric"].clip(lower=0.0),
        df["importance_value_numeric"].abs(),
    )

    df["feature_idx"] = df["feature_idx"].astype(int)
    df["fold"] = df["fold"].astype(int)
    df["protocol_norm"] = df["protocol"].map(protocol_match)

    print(f"Filtered List A: {df.shape}")
    print(f"Datasets : {sorted(df['dataset'].unique())}")
    print(f"Models   : {sorted(df['model'].unique())}")
    print(f"Protocols: {sorted(df['protocol_norm'].unique())}")
    print(f"Folds    : {sorted(df['fold'].unique())}")
    print("Rows by dataset/model:")
    print(
        df.groupby(["dataset", "model"])
        .size()
        .rename("n_rows")
        .reset_index()
        .to_string(index=False)
    )

    return df


list_a = load_list_a()


def load_list_b() -> pd.DataFrame:
    """
    Load List B: fold-discriminator feature importance.

    List B contains the ECFP4 features used by the dataset/fold detection task,
    i.e. the classifiers trained to distinguish Lo-Hi fold pairs such as
    F1_vs_F2, F1_vs_F3 and F2_vs_F3.
    """
    print_section("Loading List B")

    preferred_path = (
        PROJECT_ROOT
        / "results"
        / "results_classifier_shift_test"
        / TASK
        / "cross_dataset_listB_same_search_cv_feature_importance.csv"
    )

    if preferred_path.exists():
        path = preferred_path
    else:
        candidates = list(
            PROJECT_ROOT.glob(
                "results/**/cross_dataset_listB_same_search_cv_feature_importance.csv"
            )
        )

        if not candidates:
            raise FileNotFoundError(
                "Could not find cross_dataset_listB_same_search_cv_feature_importance.csv "
                "under results/**. Run the distribution-shift notebook first and check the saved filename."
            )

        path = sorted(candidates)[0]

    print(f"Using List B file: {path}")

    assert (
        path.name == "cross_dataset_listB_same_search_cv_feature_importance.csv"
    ), f"Wrong List B file selected: {path}"

    df = pd.read_csv(path, low_memory=False).copy()
    print(f"Raw List B: {df.shape}")

    df["dataset"] = df["dataset"].astype(str).str.lower()
    df = df[df["dataset"].isin(DATASETS_MAIN)].copy()

    df["model"] = df["model"].map(lambda x: normalize_model_name(x)).astype(str)

    if "fingerprint" in df.columns:
        df["fingerprint"] = (
            df["fingerprint"].map(lambda x: normalize_fingerprint_name(x)).astype(str)
        )
    elif "fp_type" in df.columns:
        df["fingerprint"] = (
            df["fp_type"].map(lambda x: normalize_fingerprint_name(x)).astype(str)
        )
    else:
        raise ValueError(
            f"No fingerprint/fp_type column found in List B. "
            f"Columns: {df.columns.tolist()}"
        )

    df = df[df["fingerprint"] == "ecfp4"].copy()

    if "pair" not in df.columns:
        if {"fold_a", "fold_b"}.issubset(df.columns):
            df["pair"] = df["fold_a"].astype(str) + "_vs_" + df["fold_b"].astype(str)
        elif {"subset_a", "subset_b"}.issubset(df.columns):
            df["pair"] = (
                df["subset_a"].astype(str) + "_vs_" + df["subset_b"].astype(str)
            )
        else:
            raise ValueError(
                "List B has no pair column and no fold_a/fold_b columns. "
                f"Columns: {df.columns.tolist()}"
            )

    df["pair"] = df["pair"].astype(str)

    imp_col = find_importance_col(df)
    print(f"Using importance column for List B: {imp_col}")

    df["importance_value_numeric"] = df[imp_col].fillna(0).astype(float)

    df["abs_importance"] = np.where(
        df["model"].eq("DT"),
        df["importance_value_numeric"].clip(lower=0.0),
        df["importance_value_numeric"].abs(),
    )

    df["feature_idx"] = df["feature_idx"].astype(int)

    print(f"Filtered List B: {df.shape}")
    print(f"Datasets: {sorted(df['dataset'].unique())}")
    print(f"Models:   {sorted(df['model'].unique())}")
    print(f"Pairs:    {sorted(df['pair'].unique())}")
    print("Rows by dataset/model/pair:")
    print(
        df.groupby(["dataset", "model", "pair"])
        .size()
        .rename("n_rows")
        .reset_index()
        .to_string(index=False)
    )

    return df


list_b = load_list_b()

print("\nList B loaded.")
print(f"Shape: {list_b.shape}")
display(list_b.head())

print("\nList A loaded.")
print(f"Shape: {list_a.shape}")
display(list_a.head())


Loading List A
Raw List A: (123516, 28)
Using importance column: importance_value
Filtered List A: (110592, 31)
Datasets : ['drd2', 'hiv', 'sol']
Models   : ['DT', 'LR', 'SVM']
Protocols: ['ood', 'random']
Folds    : [1, 2, 3]
Rows by dataset/model:
dataset model  n_rows
   drd2    DT   12288
   drd2    LR   12288
   drd2   SVM   12288
    hiv    DT   12288
    hiv    LR   12288
    hiv   SVM   12288
    sol    DT   12288
    sol    LR   12288
    sol   SVM   12288

Loading List B
Using List B file: /home/f.capria/drug-discovery-lohi/results/results_classifier_shift_test/hi/cross_dataset_listB_same_search_cv_feature_importance.csv
Raw List B: (61758, 27)
Using importance column for List B: permutation_importance_mean
Filtered List B: (55296, 28)
Datasets: ['drd2', 'hiv', 'sol']
Models:   ['DT', 'LR', 'SVM']
Pairs:    ['F1_vs_F2', 'F1_vs_F3', 'F2_vs_F3']
Rows by dataset/model/pair:
dataset model     pair  n_rows
   drd2    DT F1_vs_F2    2048
   drd2    DT F1_vs_F3    2048
   drd2    D

,dataset,task,pair,fold_a,fold_b,fingerprint,model,source,feature_idx,feature_name,...,permutation_eval_set,permutation_n_repeats,importance,abs_importance,importance_source,rank,coefficient,direction,normalized_importance,importance_value_numeric
0,drd2,hi,F1_vs_F2,F1,F2,ecfp4,DT,same_search_cv,1671,ecfp4_bit_1671,...,fold_discriminator_holdout,10.0,0.168424,0.168424,permutation_importance_mean,1,NaN,tree,0.200109,0.168424
1,drd2,hi,F1_vs_F2,F1,F2,ecfp4,DT,same_search_cv,29,ecfp4_bit_29,...,fold_discriminator_holdout,10.0,0.144142,0.144142,permutation_importance_mean,2,NaN,tree,0.171260,0.144142
2,drd2,hi,F1_vs_F2,F1,F2,ecfp4,DT,same_search_cv,231,ecfp4_bit_231,...,fold_discriminator_holdout,10.0,0.073325,0.073325,permutation_importance_mean,3,NaN,tree,0.087120,0.073325
3,drd2,hi,F1_vs_F2,F1,F2,ecfp4,DT,same_search_cv,197,ecfp4_bit_197,...,fold_discriminator_holdout,10.0,0.058826,0.058826,permutation_importance_mean,4,NaN,tree,0.069893,0.058826
4,drd2,hi,F1_vs_F2,F1,F2,ecfp4,DT,same_search_cv,1303,ecfp4_bit_1303,...,fold_discriminator_holdout,10.0,0.055321,0.055321,permutation_importance_mean,5,NaN,tree,0.065728,0.055321



List A loaded.
Shape: (110592, 31)


,feature_idx,feature_name,tree_importance,minimum_depth,used_in_tree,normalized_tree_importance,rank_tree_importance,permutation_importance_mean,permutation_importance_std,permutation_scoring,...,protocol,fold,raw_weight,abs_weight,normalized_abs_importance,direction,rank_abs_weight,importance_value_numeric,abs_importance,protocol_norm
0,1145,ecfp4_bit_1145,0.051899,5.0,True,0.051899,2.0,0.034229,0.004964,average_precision,...,OOD holdout,1,NaN,NaN,NaN,NaN,NaN,0.034229,0.034229,ood
1,1948,ecfp4_bit_1948,0.010433,11.0,True,0.010433,27.0,0.012609,0.005989,average_precision,...,OOD holdout,1,NaN,NaN,NaN,NaN,NaN,0.012609,0.012609,ood
2,592,ecfp4_bit_592,0.027457,3.0,True,0.027457,9.0,0.011152,0.003453,average_precision,...,OOD holdout,1,NaN,NaN,NaN,NaN,NaN,0.011152,0.011152,ood
3,942,ecfp4_bit_942,0.013437,9.0,True,0.013437,17.0,0.009500,0.001523,average_precision,...,OOD holdout,1,NaN,NaN,NaN,NaN,NaN,0.009500,0.009500,ood
4,1917,ecfp4_bit_1917,0.015700,5.0,True,0.015700,15.0,0.008028,0.002487,average_precision,...,OOD holdout,1,NaN,NaN,NaN,NaN,NaN,0.008028,0.008028,ood


In [8]:
print_section("Model-wise analysis")

print(
    "The Tanimoto restricted-space analysis is now computed model-wise. "
    "No single best ECFP4 model is selected per dataset."
)

print(f"Models included: {MODELS}")


Model-wise analysis
The Tanimoto restricted-space analysis is now computed model-wise. No single best ECFP4 model is selected per dataset.
Models included: ['DT', 'LR', 'SVM']


In [9]:
def topk_bits_global(
    list_a: pd.DataFrame, dataset: str, model: str, k: int
) -> np.ndarray:
    """Top-k ECFP4 bits pooled across protocols and folds for one (dataset, model)."""
    sub = list_a[(list_a["dataset"] == dataset) & (list_a["model"] == model)]
    if sub.empty:
        return np.array([], dtype=int)
    agg = (
        sub.groupby("feature_idx")["abs_importance"].mean().sort_values(ascending=False)
    )
    return agg.head(k).index.to_numpy(dtype=int)


def topk_bits_fold_protocol(
    list_a: pd.DataFrame,
    dataset: str,
    model: str,
    fold: int,
    protocol: str,
    k: int,
) -> np.ndarray:
    """Top-k ECFP4 bits for (dataset, model, fold, protocol) — fold/protocol-aware."""
    protocol = protocol_match(protocol)
    sub = list_a[
        (list_a["dataset"] == dataset)
        & (list_a["model"] == model)
        & (list_a["fold"].astype(int) == int(fold))
        & (list_a["protocol_norm"] == protocol)
    ]
    if sub.empty:
        return np.array([], dtype=int)
    agg = (
        sub.groupby("feature_idx")["abs_importance"].mean().sort_values(ascending=False)
    )
    return agg.head(k).index.to_numpy(dtype=int)


def random_topk_bits(dataset: str, pair: str, k: int, repeat: int) -> np.ndarray:
    """Random subset of k ECFP4 bits, deterministic by (dataset, pair, k, repeat)."""
    rng = local_rng("random_bits", dataset, pair, k, repeat)
    return rng.choice(EXPECTED_ECFP4_BITS, size=k, replace=False)


def topk_bits_list_b(
    list_b: pd.DataFrame,
    dataset: str,
    model: str,
    pair: str,
    k: int,
) -> np.ndarray:
    """
    Top-k ECFP4 bits from List B for one dataset, model and fold pair.

    These bits come from the dataset/fold detection task, not from the
    activity-prediction task.
    """
    sub = list_b[
        (list_b["dataset"] == dataset)
        & (list_b["model"] == model)
        & (list_b["pair"] == pair)
    ]

    if sub.empty:
        warnings.warn(
            f"No List B bits found for dataset={dataset}, model={model}, pair={pair}."
        )
        return np.array([], dtype=int)

    agg = (
        sub.groupby("feature_idx")["abs_importance"].mean().sort_values(ascending=False)
    )

    return agg.head(k).index.to_numpy(dtype=int)


def build_selected_bits_table(
    list_a: pd.DataFrame, list_b: pd.DataFrame
) -> pd.DataFrame:
    """
    Save the selected top-k ECFP4 bit sets used in the restricted-space analysis.

    This table is useful for fold-wise plots of:
    - OOD activity vs random-shuffle activity overlap;
    - activity vs dataset-detection overlap;
    - overlap growth as a function of k.
    """
    rows = []

    for dataset in DATASETS:
        for model in MODELS:
            for fold_a, fold_b in PAIRS:
                pair = f"{fold_a}_vs_{fold_b}"
                outer_fold = PAIR_TO_OUTER_FOLD[pair]

                for k in K_VALUES:
                    bit_sets = {
                        "activity_global": (
                            topk_bits_global(list_a, dataset, model, k),
                            "pooled",
                            np.nan,
                        ),
                        "activity_ood": (
                            topk_bits_fold_protocol(
                                list_a, dataset, model, outer_fold, "ood", k
                            ),
                            "ood",
                            outer_fold,
                        ),
                        "activity_random_shuffle": (
                            topk_bits_fold_protocol(
                                list_a, dataset, model, outer_fold, "random", k
                            ),
                            "random",
                            outer_fold,
                        ),
                        "dataset_detection": (
                            topk_bits_list_b(list_b, dataset, model, pair, k),
                            "same_search_cv",
                            outer_fold,
                        ),
                    }

                    for bit_source, (bits, protocol, activity_fold) in bit_sets.items():
                        bits = [int(b) for b in bits]
                        rows.append(
                            {
                                "dataset": dataset,
                                "dataset_label": DATASET_LABELS.get(dataset, dataset),
                                "model": model,
                                "pair": pair,
                                "outer_fold": outer_fold,
                                "k": k,
                                "bit_source": bit_source,
                                "activity_protocol": protocol,
                                "activity_fold": activity_fold,
                                "n_bits": len(bits),
                                "bits_json": json.dumps(bits),
                            }
                        )

    out = pd.DataFrame(rows)

    out_path = OUT_ROOT / "fold_distance_selected_bits_modelwise.csv"
    out.to_csv(out_path, index=False)

    print(f"Saved selected bits table: {out_path} ({len(out)} rows)")
    return out


selected_bits = build_selected_bits_table(list_a, list_b)
display(selected_bits.head())

Saved selected bits table: /home/f.capria/drug-discovery-lohi/results/results_fold_distance_tanimoto/hi/fold_distance_selected_bits_modelwise.csv (864 rows)


,dataset,dataset_label,model,pair,outer_fold,k,bit_source,activity_protocol,activity_fold,n_bits,bits_json
0,drd2,DRD2,DT,F1_vs_F2,1,10,activity_global,pooled,NaN,10,"[1145, 2000, 592, 1911, 656, 1791, 561, 866, 1..."
1,drd2,DRD2,DT,F1_vs_F2,1,10,activity_ood,ood,1.0,10,"[1145, 1948, 592, 942, 1917, 2000, 1034, 1160,..."
2,drd2,DRD2,DT,F1_vs_F2,1,10,activity_random_shuffle,random,1.0,10,"[592, 1145, 866, 562, 119, 841, 80, 1199, 1518..."
3,drd2,DRD2,DT,F1_vs_F2,1,10,dataset_detection,same_search_cv,1.0,10,"[1671, 29, 231, 197, 1303, 486, 1292, 1295, 20..."
4,drd2,DRD2,DT,F1_vs_F2,1,20,activity_global,pooled,NaN,20,"[1145, 2000, 592, 1911, 656, 1791, 561, 866, 1..."


In [10]:
def restrict_to_bits(X: np.ndarray, bits: np.ndarray) -> np.ndarray:
    if len(bits) == 0:
        return np.zeros((X.shape[0], 0), dtype=X.dtype)
    return X[:, bits]


def wasserstein_nd_optional(
    XA: np.ndarray,
    XB: np.ndarray,
    n: int,
    dataset: str,
    pair: str,
    space: str,
) -> float:
    """Optional ND Wasserstein-1 distance with Euclidean ground metric. Disabled by default."""
    if not RUN_WASSERSTEIN or not _HAS_WASSERSTEIN:
        return np.nan
    rng = local_rng("wasserstein", dataset, pair, space)
    if XA.shape[0] > n:
        XA = XA[rng.choice(XA.shape[0], size=n, replace=False)]
    if XB.shape[0] > n:
        XB = XB[rng.choice(XB.shape[0], size=n, replace=False)]
    try:
        return float(
            wasserstein_distance_nd(XA.astype(np.float32), XB.astype(np.float32))
        )
    except Exception as exc:
        warnings.warn(f"Wasserstein failed for {dataset}|{pair}|{space}: {exc}")
        return np.nan


print("Top-k bit selection and optional Wasserstein functions defined.")

Top-k bit selection and optional Wasserstein functions defined.


In [11]:
DIST_ROW_KEYS = [
    "dataset",
    "dataset_label",
    "pair",
    "space",
    "k",
    "model",
    "bit_source",
    "activity_protocol",
    "activity_fold",
    "bit_repeat",
    "bits_used",
    "nn_sym_distance",
    "nn_A_to_B_mean",
    "nn_B_to_A_mean",
    "wasserstein_nd",
    "valid_molecule_fraction",
    "n_valid_molecules",
    "n_total_molecules",
    "pairwise_distance",
    "valid_pair_fraction",
    "n_valid_pairs",
    "n_total_pairs",
    "pairwise_mode",
    "outer_fold",
]


def _base_row(
    dataset,
    pair,
    space,
    k,
    model,
    bit_source,
    activity_protocol,
    activity_fold,
    bit_repeat,
    bits_used,
):
    return {
        "dataset": dataset,
        "dataset_label": DATASET_LABELS.get(dataset, dataset),
        "pair": pair,
        "outer_fold": PAIR_TO_OUTER_FOLD.get(pair, np.nan),
        "space": space,
        "k": int(k),
        "model": model,
        "bit_source": bit_source,
        "activity_protocol": activity_protocol,
        "activity_fold": activity_fold,
        "bit_repeat": bit_repeat,
        "bits_used": int(bits_used),
    }


def add_distance_row(
    dist_rows,
    hist_rows,
    dataset,
    pair,
    XA,
    XB,
    model,
):
    """Full ECFP4 space row."""
    space = "full_ecfp4"
    bit_source = "full_ecfp4"
    nn = nn_distances(XA, XB)
    pw = complete_pairwise_distance(
        XA,
        XB,
        dataset=dataset,
        pair=pair,
        space=space,
    )
    wd = wasserstein_nd_optional(XA, XB, W_SUBSAMPLE, dataset, pair, space)

    n_valid_mol = nn["n_valid_a"] + nn["n_valid_b"]
    n_total_mol = nn["n_total_a"] + nn["n_total_b"]

    row = _base_row(
        dataset,
        pair,
        space,
        EXPECTED_ECFP4_BITS,
        model,
        bit_source,
        np.nan,
        np.nan,
        np.nan,
        EXPECTED_ECFP4_BITS,
    )
    row.update(
        {
            "nn_sym_distance": nn["nn_sym_distance"],
            "nn_A_to_B_mean": nn["nn_A_to_B_mean"],
            "nn_B_to_A_mean": nn["nn_B_to_A_mean"],
            "wasserstein_nd": wd,
            "valid_molecule_fraction": n_valid_mol / max(n_total_mol, 1),
            "n_valid_molecules": n_valid_mol,
            "n_total_molecules": n_total_mol,
            "pairwise_distance": pw["pairwise_distance"],
            "valid_pair_fraction": pw["valid_pair_fraction"],
            "n_valid_pairs": pw["n_valid_pairs"],
            "n_total_pairs": pw["n_total_pairs"],
            "pairwise_mode": pw["pairwise_mode"],
        }
    )
    dist_rows.append(row)

    for d in nn["nn_AB_array"]:
        hist_rows.append(
            {
                "dataset": dataset,
                "dataset_label": DATASET_LABELS.get(dataset, dataset),
                "model": model,
                "pair": pair,
                "outer_fold": PAIR_TO_OUTER_FOLD.get(pair, np.nan),
                "space": space,
                "k": EXPECTED_ECFP4_BITS,
                "bit_source": bit_source,
                "activity_protocol": np.nan,
                "activity_fold": np.nan,
                "bit_repeat": np.nan,
                "bits_used": EXPECTED_ECFP4_BITS,
                "direction": "A_to_B",
                "nn_distance": float(d),
            }
        )
    for d in nn["nn_BA_array"]:
        hist_rows.append(
            {
                "dataset": dataset,
                "dataset_label": DATASET_LABELS.get(dataset, dataset),
                "model": model,
                "pair": pair,
                "outer_fold": PAIR_TO_OUTER_FOLD.get(pair, np.nan),
                "space": space,
                "k": EXPECTED_ECFP4_BITS,
                "bit_source": bit_source,
                "activity_protocol": np.nan,
                "activity_fold": np.nan,
                "bit_repeat": np.nan,
                "bits_used": EXPECTED_ECFP4_BITS,
                "direction": "B_to_A",
                "nn_distance": float(d),
            }
        )


def add_restricted_distance_row(
    dist_rows,
    hist_rows,
    dataset,
    pair,
    XA,
    XB,
    bits,
    k,
    model,
    space,
    bit_source,
    activity_protocol,
    activity_fold,
    bit_repeat,
    store_hist: bool,
):
    """Top-k restricted space row."""
    if len(bits) == 0 and bit_source in {
        "activity_global",
        "activity_ood",
        "activity_random_shuffle",
        "dataset_detection",
    }:
        warnings.warn(
            f"No restricted-space bits found for {dataset} | {pair} | {space} | "
            f"model={model} | bit_source={bit_source}. "
            "Distances will be NaN or based on empty restricted fingerprints."
        )
    XA_r = restrict_to_bits(XA, bits)
    XB_r = restrict_to_bits(XB, bits)

    nn = nn_distances(XA_r, XB_r)
    pw = complete_pairwise_distance(
        XA_r,
        XB_r,
        dataset=dataset,
        pair=pair,
        space=space,
    )
    wd = wasserstein_nd_optional(XA_r, XB_r, W_SUBSAMPLE, dataset, pair, space)

    n_valid_mol = nn["n_valid_a"] + nn["n_valid_b"]
    n_total_mol = nn["n_total_a"] + nn["n_total_b"]

    row = _base_row(
        dataset,
        pair,
        space,
        k,
        model,
        bit_source,
        activity_protocol,
        activity_fold,
        bit_repeat,
        len(bits),
    )
    row.update(
        {
            "nn_sym_distance": nn["nn_sym_distance"],
            "nn_A_to_B_mean": nn["nn_A_to_B_mean"],
            "nn_B_to_A_mean": nn["nn_B_to_A_mean"],
            "wasserstein_nd": wd,
            "valid_molecule_fraction": n_valid_mol / max(n_total_mol, 1),
            "n_valid_molecules": n_valid_mol,
            "n_total_molecules": n_total_mol,
            "pairwise_distance": pw["pairwise_distance"],
            "valid_pair_fraction": pw["valid_pair_fraction"],
            "n_valid_pairs": pw["n_valid_pairs"],
            "n_total_pairs": pw["n_total_pairs"],
            "pairwise_mode": pw["pairwise_mode"],
        }
    )
    dist_rows.append(row)

    if store_hist:
        for d in nn["nn_AB_array"]:
            hist_rows.append(
                {
                    "dataset": dataset,
                    "dataset_label": DATASET_LABELS.get(dataset, dataset),
                    "model": model,
                    "pair": pair,
                    "outer_fold": PAIR_TO_OUTER_FOLD.get(pair, np.nan),
                    "space": space,
                    "k": k,
                    "bit_source": bit_source,
                    "activity_protocol": activity_protocol,
                    "activity_fold": activity_fold,
                    "bit_repeat": bit_repeat,
                    "bits_used": len(bits),
                    "direction": "A_to_B",
                    "nn_distance": float(d),
                }
            )
        for d in nn["nn_BA_array"]:
            hist_rows.append(
                {
                    "dataset": dataset,
                    "dataset_label": DATASET_LABELS.get(dataset, dataset),
                    "model": model,
                    "pair": pair,
                    "outer_fold": PAIR_TO_OUTER_FOLD.get(pair, np.nan),
                    "space": space,
                    "k": k,
                    "bit_source": bit_source,
                    "activity_protocol": activity_protocol,
                    "activity_fold": activity_fold,
                    "bit_repeat": bit_repeat,
                    "bits_used": len(bits),
                    "direction": "B_to_A",
                    "nn_distance": float(d),
                }
            )


print("Distance row-builder functions defined.")

Distance row-builder functions defined.


### Compute dataset distances


In [12]:
def compute_dataset_distances(
    dataset: str,
    fps: dict[str, tuple[np.ndarray, pd.DataFrame]],
    list_a: pd.DataFrame,
    list_b: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    print_section(f"Computing distances: {dataset.upper()}-{TASK}")

    dist_rows: list[dict] = []
    hist_rows: list[dict] = []

    for model in MODELS:
        print(f"\n### Model: {model}")

        for fold_a, fold_b in PAIRS:
            pair = f"{fold_a}_vs_{fold_b}"
            outer_fold = PAIR_TO_OUTER_FOLD[pair]

            XA, _ = fps[fold_a]
            XB, _ = fps[fold_b]

            print(
                f"\n  Pair {pair} (outer fold {outer_fold}) | "
                f"model={model}: nA={XA.shape[0]} nB={XB.shape[0]}"
            )

            # Full ECFP4 space.
            # This is repeated per model to keep the output table aligned with
            # the restricted-space model-wise rows.
            add_distance_row(
                dist_rows,
                hist_rows,
                dataset,
                pair,
                XA,
                XB,
                model,
            )

            # Restricted top-k spaces
            for k in K_VALUES:
                print(
                    f"    k={k}: activity spaces + dataset-detection space "
                    "+ random-bit baseline"
                )

                # Global activity bits (pooled across protocols and folds)
                bits_g = topk_bits_global(list_a, dataset, model, k)
                print(f"      activity_global: {len(bits_g)} bits")
                add_restricted_distance_row(
                    dist_rows,
                    hist_rows,
                    dataset,
                    pair,
                    XA,
                    XB,
                    bits=bits_g,
                    k=k,
                    model=model,
                    space=f"activity_global_top{k}",
                    bit_source="activity_global",
                    activity_protocol="pooled",
                    activity_fold=np.nan,
                    bit_repeat=np.nan,
                    store_hist=True,
                )

                # OOD activity bits, fold-aware
                bits_ood = topk_bits_fold_protocol(
                    list_a, dataset, model, outer_fold, "ood", k
                )
                print(f"      activity_ood fold={outer_fold}: {len(bits_ood)} bits")
                add_restricted_distance_row(
                    dist_rows,
                    hist_rows,
                    dataset,
                    pair,
                    XA,
                    XB,
                    bits=bits_ood,
                    k=k,
                    model=model,
                    space=f"activity_ood_top{k}",
                    bit_source="activity_ood",
                    activity_protocol="ood",
                    activity_fold=outer_fold,
                    bit_repeat=np.nan,
                    store_hist=True,
                )

                # Random-shuffle activity bits, fold-aware
                bits_rnd = topk_bits_fold_protocol(
                    list_a, dataset, model, outer_fold, "random", k
                )
                print(
                    f"      activity_random_shuffle fold={outer_fold}: "
                    f"{len(bits_rnd)} bits"
                )
                add_restricted_distance_row(
                    dist_rows,
                    hist_rows,
                    dataset,
                    pair,
                    XA,
                    XB,
                    bits=bits_rnd,
                    k=k,
                    model=model,
                    space=f"activity_random_shuffle_top{k}",
                    bit_source="activity_random_shuffle",
                    activity_protocol="random",
                    activity_fold=outer_fold,
                    bit_repeat=np.nan,
                    store_hist=True,
                )

                # Dataset/fold detection bits, fold-pair-aware
                bits_det = topk_bits_list_b(
                    list_b=list_b,
                    dataset=dataset,
                    model=model,
                    pair=pair,
                    k=k,
                )
                print(f"      dataset_detection {pair}: {len(bits_det)} bits")
                add_restricted_distance_row(
                    dist_rows,
                    hist_rows,
                    dataset,
                    pair,
                    XA,
                    XB,
                    bits=bits_det,
                    k=k,
                    model=model,
                    space=f"dataset_detection_top{k}",
                    bit_source="dataset_detection",
                    activity_protocol="same_search_cv",
                    activity_fold=outer_fold,
                    bit_repeat=np.nan,
                    store_hist=True,
                )

                # Random-bit baseline (negative control for dimensionality)
                for r in range(N_RANDOM_BIT_REPEATS):
                    if (
                        r == 0
                        or (r + 1) % PRINT_EVERY_RANDOM_REPEAT == 0
                        or r == N_RANDOM_BIT_REPEATS - 1
                    ):
                        print(
                            f"      random_bits repeat {r + 1}/{N_RANDOM_BIT_REPEATS}"
                        )

                    bits_rb = random_topk_bits(dataset, pair, k, r)
                    add_restricted_distance_row(
                        dist_rows,
                        hist_rows,
                        dataset,
                        pair,
                        XA,
                        XB,
                        bits=bits_rb,
                        k=k,
                        model=model,
                        space=f"random_bits_top{k}",
                        bit_source="random_bits",
                        activity_protocol=np.nan,
                        activity_fold=np.nan,
                        bit_repeat=r,
                        store_hist=False,
                    )

    dist_df = pd.DataFrame(dist_rows)
    hist_df = pd.DataFrame(hist_rows)

    n_random_bit_rows = (dist_df["bit_source"] == "random_bits").sum()
    expected_rb = len(MODELS) * len(PAIRS) * len(K_VALUES) * N_RANDOM_BIT_REPEATS

    print(f"\n  Dataset {dataset}: dist_rows={len(dist_df)}, hist_rows={len(hist_df)}")
    print(
        f"  random_bits rows = {n_random_bit_rows} (expected {expected_rb}: "
        f"{len(MODELS)} models × {len(PAIRS)} pairs × "
        f"{len(K_VALUES)} k × {N_RANDOM_BIT_REPEATS} repeats)"
    )

    expected_total = (
        len(MODELS)
        * len(PAIRS)
        * (1 + len(K_VALUES) * (1 + 1 + 1 + 1 + N_RANDOM_BIT_REPEATS))
    )

    if len(dist_df) != expected_total:
        warnings.warn(
            f"{dataset}: got {len(dist_df)} dist rows, expected {expected_total}"
        )

    return dist_df, hist_df

### Compute distances dataset by dataset

In [13]:
dist_parts = []
hist_parts = []

for ds in DATASETS:
    print_section(f"Running dataset: {ds.upper()}")

    fps = load_subset_fps(ds)

    t0 = time.time()

    dist_df, hist_df = compute_dataset_distances(
        dataset=ds,
        fps=fps,
        list_a=list_a,
        list_b=list_b,
    )

    elapsed = time.time() - t0

    dist_parts.append(dist_df)
    hist_parts.append(hist_df)

    dataset_dist_path = OUT_ROOT / f"fold_distance_summary_{ds}.csv"
    dataset_hist_path = OUT_ROOT / f"fold_distance_nn_distributions_{ds}.csv"

    dist_df.to_csv(dataset_dist_path, index=False)
    hist_df.to_csv(dataset_hist_path, index=False)

    print(
        f"Saved {ds}: dist_rows={len(dist_df)}, hist_rows={len(hist_df)}, "
        f"time={elapsed / 60:.1f} min"
    )
    print(f"  {dataset_dist_path}")
    print(f"  {dataset_hist_path}")


Running dataset: DRD2

Loading subsets: DRD2-hi
  F1 (test_3.csv): raw=1191, valid=1191, invalid=0, n_bits=2048, time=0.6s
  F2 (test_2.csv): raw=1194, valid=1194, invalid=0, n_bits=2048, time=0.6s
  F3 (test_1.csv): raw=1190, valid=1190, invalid=0, n_bits=2048, time=0.5s

Computing distances: DRD2-hi

### Model: DT

  Pair F1_vs_F2 (outer fold 1) | model=DT: nA=1191 nB=1194
    k=10: activity spaces + dataset-detection space + random-bit baseline
      activity_global: 10 bits
      activity_ood fold=1: 10 bits
      activity_random_shuffle fold=1: 10 bits
      dataset_detection F1_vs_F2: 10 bits
      random_bits repeat 1/30
      random_bits repeat 5/30
      random_bits repeat 10/30
      random_bits repeat 15/30
      random_bits repeat 20/30
      random_bits repeat 25/30
      random_bits repeat 30/30
    k=20: activity spaces + dataset-detection space + random-bit baseline
      activity_global: 20 bits
      activity_ood fold=1: 20 bits
      activity_random_shuffle fold=1: 

[18:41:49] Explicit valence for atom # 4 Al, 9, is greater than permitted
Invalid SMILES at index 6699: CC1=C2[OH+][AlH3-3]34([O+]=C2C=CN1C)([O+]=C1C=CN(C)C(C)=C1[OH+]3)[O+]=C1C=CN(C)C


  F1 (test_3.csv): raw=7848, valid=7847, invalid=1, n_bits=2048, time=3.2s


[18:41:51] Explicit valence for atom # 16 Al, 9, is greater than permitted
Invalid SMILES at index 2458: Oc1ccc(C2Oc3cc(O)cc4c3C(=[O+][AlH3-3]35([O+]=C6c7c(cc(O)cc7[OH+]3)OC(c3ccc(O)cc3
[18:41:52] Explicit valence for atom # 5 B, 5, is greater than permitted
Invalid SMILES at index 5054: Cc1ccc([B-2]2(c3ccc(C)cc3)=NCCO2)cc1
[18:41:53] WARNING: not removing hydrogen atom without neighbors
[18:41:53] WARNING: not removing hydrogen atom without neighbors


  F2 (test_2.csv): raw=7848, valid=7846, invalid=2, n_bits=2048, time=3.2s


[18:41:55] Explicit valence for atom # 3 Al, 6, is greater than permitted
Invalid SMILES at index 4822: O=C1O[Al]23(OC1=O)(OC(=O)C(=O)O2)OC(=O)C(=O)O3
[18:41:55] Explicit valence for atom # 12 Al, 7, is greater than permitted
Invalid SMILES at index 7350: CC(c1cccs1)=[N+]1[N-]C(N)=[S+][AlH3-]12[OH+]B(c1ccccc1)[OH+]2
[18:41:55] Explicit valence for atom # 13 Al, 7, is greater than permitted
Invalid SMILES at index 7351: CC(c1ccccn1)=[N+]1[N-]C(N)=[S+][AlH3-]12[OH+]B(c1ccccc1)[OH+]2
[18:41:55] Explicit valence for atom # 6 Ge, 5, is greater than permitted
Invalid SMILES at index 7588: [Na+].c1ccc([SH+][GeH2+]2[SH+]c3ccccc3[SH+]2)c([SH+][GeH2+]2[SH+]c3ccccc3[SH+]2)


  F3 (test_1.csv): raw=7847, valid=7843, invalid=4, n_bits=2048, time=2.6s

Computing distances: HIV-hi

### Model: DT

  Pair F1_vs_F2 (outer fold 1) | model=DT: nA=7847 nB=7846
    k=10: activity spaces + dataset-detection space + random-bit baseline
      activity_global: 10 bits
      activity_ood fold=1: 10 bits
      activity_random_shuffle fold=1: 10 bits
      dataset_detection F1_vs_F2: 10 bits
      random_bits repeat 1/30
      random_bits repeat 5/30
      random_bits repeat 10/30
      random_bits repeat 15/30
      random_bits repeat 20/30
      random_bits repeat 25/30
      random_bits repeat 30/30
    k=20: activity spaces + dataset-detection space + random-bit baseline
      activity_global: 20 bits
      activity_ood fold=1: 20 bits
      activity_random_shuffle fold=1: 20 bits
      dataset_detection F1_vs_F2: 20 bits
      random_bits repeat 1/30
      random_bits repeat 5/30
      random_bits repeat 10/30
      random_bits repeat 15/30
      random_bits repeat 20/

### Concatenate all datasets and save global tables

In [14]:
if not dist_parts:
    raise RuntimeError("No dataset produced distance rows.")

dist_all = pd.concat(dist_parts, ignore_index=True)
hist_all = pd.concat(hist_parts, ignore_index=True) if hist_parts else pd.DataFrame()

# Verify dist_all contains all required columns from the spec
missing_cols = set(DIST_ROW_KEYS) - set(dist_all.columns)
assert not missing_cols, f"dist_all missing required columns: {missing_cols}"

dist_path = OUT_ROOT / "fold_distance_summary.csv"
hist_path = OUT_ROOT / "fold_distance_nn_distributions.csv"

dist_all.to_csv(dist_path, index=False)
hist_all.to_csv(hist_path, index=False)

print("Saved global tables.")
print(f"dist_all: {dist_all.shape} -> {dist_path}")
print(f"hist_all: {hist_all.shape} -> {hist_path}")

display(dist_all.head())

Saved global tables.
dist_all: (7371, 24) -> /home/f.capria/drug-discovery-lohi/results/results_fold_distance_tanimoto/hi/fold_distance_summary.csv
hist_all: (4672922, 14) -> /home/f.capria/drug-discovery-lohi/results/results_fold_distance_tanimoto/hi/fold_distance_nn_distributions.csv


,dataset,dataset_label,pair,outer_fold,space,k,model,bit_source,activity_protocol,activity_fold,...,nn_B_to_A_mean,wasserstein_nd,valid_molecule_fraction,n_valid_molecules,n_total_molecules,pairwise_distance,valid_pair_fraction,n_valid_pairs,n_total_pairs,pairwise_mode
0,drd2,DRD2,F1_vs_F2,1,full_ecfp4,2048,DT,full_ecfp4,NaN,NaN,...,0.699229,NaN,1.000000,2385,2385,0.852592,1.000000,1422054,1422054,complete
1,drd2,DRD2,F1_vs_F2,1,activity_global_top10,10,DT,activity_global,pooled,NaN,...,0.043879,NaN,0.927044,2211,2385,0.708013,0.994881,1414774,1422054,complete
2,drd2,DRD2,F1_vs_F2,1,activity_ood_top10,10,DT,activity_ood,ood,1.0,...,0.045716,NaN,0.906080,2161,2385,0.747359,0.991433,1409871,1422054,complete
3,drd2,DRD2,F1_vs_F2,1,activity_random_shuffle_top10,10,DT,activity_random_shuffle,random,1.0,...,0.018830,NaN,0.989099,2359,2385,0.546577,0.999882,1421886,1422054,complete
4,drd2,DRD2,F1_vs_F2,1,dataset_detection_top10,10,DT,dataset_detection,same_search_cv,1.0,...,0.219711,NaN,0.949266,2264,2385,0.743797,0.998169,1419450,1422054,complete


In [15]:
def print_output_diagnostics(dist_all: pd.DataFrame, hist_all: pd.DataFrame):
    print_section("Output diagnostics")

    print(f"dist_all shape: {dist_all.shape}")
    print(f"hist_all shape: {hist_all.shape}")

    if len(dist_all) == 0:
        raise RuntimeError("dist_all is empty — distance computation produced no rows.")

    print("\nRows by dataset / model / bit_source:")
    print(
        dist_all.groupby(["dataset", "model", "bit_source"])
        .size()
        .rename("n")
        .reset_index()
        .to_string(index=False)
    )

    expected_per_dataset = (
        len(MODELS) * len(PAIRS) * (1 + len(K_VALUES) * (4 + N_RANDOM_BIT_REPEATS))
    )

    expected_total = expected_per_dataset * len(DATASETS)

    print("\nExpected row count:")
    print(f"  per dataset: {expected_per_dataset}")
    print(f"  total:       {expected_total}")
    print(f"  observed:    {len(dist_all)}")

    unexpected_datasets = set(dist_all["dataset"].unique()) - set(DATASETS)
    assert not unexpected_datasets, f"Unexpected datasets found: {unexpected_datasets}"

    assert "kdr" not in set(
        dist_all["dataset"].astype(str).str.lower()
    ), "KDR leaked into the analysis."

    for ds in DATASETS:
        got = int((dist_all["dataset"] == ds).sum())
        flag = "OK" if got == expected_per_dataset else "MISSING"
        print(f"  {ds}: got {got}, expected {expected_per_dataset} [{flag}]")

    if len(dist_all) != expected_total:
        raise RuntimeError(
            f"Unexpected total row count: got {len(dist_all)}, expected {expected_total}"
        )

    expected_bit_sources = {
        "full_ecfp4",
        "activity_global",
        "activity_ood",
        "activity_random_shuffle",
        "dataset_detection",
        "random_bits",
    }

    missing_sources = expected_bit_sources - set(dist_all["bit_source"].unique())
    if missing_sources:
        raise RuntimeError(f"Missing bit sources: {missing_sources}")

    missing_models = set(MODELS) - set(dist_all["model"].unique())
    if missing_models:
        raise RuntimeError(f"Missing models: {missing_models}")

    if "pairwise_distance" not in dist_all.columns:
        raise RuntimeError("Missing main metric column: pairwise_distance")

    if dist_all["pairwise_distance"].notna().sum() == 0:
        raise RuntimeError("All pairwise_distance values are NaN — pipeline failed.")

    print(
        f"\nFinite pairwise_distance: "
        f"{dist_all['pairwise_distance'].notna().sum()}/{len(dist_all)}"
    )

    if "valid_pair_fraction" in dist_all.columns:
        print("\nValid pair fraction summary:")
        print(
            dist_all["valid_pair_fraction"]
            .describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
            .to_string()
        )

    if "valid_molecule_fraction" in dist_all.columns:
        print("\nValid molecule fraction summary:")
        print(
            dist_all["valid_molecule_fraction"]
            .describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
            .to_string()
        )

    print("\nOK: diagnostics passed.")


print_output_diagnostics(dist_all, hist_all)


Output diagnostics
dist_all shape: (7371, 24)
hist_all shape: (4672922, 14)

Rows by dataset / model / bit_source:
dataset model              bit_source   n
   drd2    DT         activity_global  24
   drd2    DT            activity_ood  24
   drd2    DT activity_random_shuffle  24
   drd2    DT       dataset_detection  24
   drd2    DT              full_ecfp4   3
   drd2    DT             random_bits 720
   drd2    LR         activity_global  24
   drd2    LR            activity_ood  24
   drd2    LR activity_random_shuffle  24
   drd2    LR       dataset_detection  24
   drd2    LR              full_ecfp4   3
   drd2    LR             random_bits 720
   drd2   SVM         activity_global  24
   drd2   SVM            activity_ood  24
   drd2   SVM activity_random_shuffle  24
   drd2   SVM       dataset_detection  24
   drd2   SVM              full_ecfp4   3
   drd2   SVM             random_bits 720
    hiv    DT         activity_global  24
    hiv    DT            activity_ood  24
  

In [16]:
def build_summary(dist_all: pd.DataFrame) -> pd.DataFrame:
    print_section("Building final summary")

    summary = dist_all.groupby(
        [
            "dataset",
            "dataset_label",
            "model",
            "pair",
            "outer_fold",
            "k",
            "bit_source",
            "activity_protocol",
            "activity_fold",
        ],
        dropna=False,
        as_index=False,
    ).agg(
        pairwise_distance_mean=("pairwise_distance", "mean"),
        pairwise_distance_std=("pairwise_distance", "std"),
        valid_molecule_fraction_mean=("valid_molecule_fraction", "mean"),
        valid_pair_fraction_mean=("valid_pair_fraction", "mean"),
        n_rows=("pairwise_distance", "size"),
    )

    # Full ECFP4 reference for each dataset/model/pair/fold.
    full_ref = summary[summary["bit_source"] == "full_ecfp4"][
        [
            "dataset",
            "model",
            "pair",
            "outer_fold",
            "pairwise_distance_mean",
        ]
    ].rename(columns={"pairwise_distance_mean": "full_pairwise_distance_mean"})

    summary = summary.merge(
        full_ref,
        on=["dataset", "model", "pair", "outer_fold"],
        how="left",
    )

    summary["delta_minus_full_pairwise"] = (
        summary["pairwise_distance_mean"] - summary["full_pairwise_distance_mean"]
    )

    # Random-bit reference for each dataset/model/pair/fold/k.
    random_ref = summary[summary["bit_source"] == "random_bits"][
        [
            "dataset",
            "model",
            "pair",
            "outer_fold",
            "k",
            "pairwise_distance_mean",
        ]
    ].rename(columns={"pairwise_distance_mean": "random_bits_pairwise_distance_mean"})

    summary = summary.merge(
        random_ref,
        on=["dataset", "model", "pair", "outer_fold", "k"],
        how="left",
    )

    summary["delta_minus_random_bits_pairwise"] = (
        summary["pairwise_distance_mean"]
        - summary["random_bits_pairwise_distance_mean"]
    )

    summary = summary.sort_values(
        [
            "dataset",
            "model",
            "outer_fold",
            "k",
            "bit_source",
            "activity_protocol",
        ],
        na_position="last",
    ).reset_index(drop=True)

    out_path = OUT_ROOT / "fold_distance_activity_detection_vs_random_bits_summary.csv"
    summary.to_csv(out_path, index=False)

    print(f"Saved summary: {out_path} ({len(summary)} rows)")

    display_cols = [
        "dataset",
        "model",
        "pair",
        "outer_fold",
        "k",
        "bit_source",
        "activity_protocol",
        "pairwise_distance_mean",
        "full_pairwise_distance_mean",
        "delta_minus_full_pairwise",
        "random_bits_pairwise_distance_mean",
        "delta_minus_random_bits_pairwise",
        "valid_molecule_fraction_mean",
        "valid_pair_fraction_mean",
    ]

    print("\nSummary preview:")
    print(summary[display_cols].head(30).to_string(index=False))

    return summary


final_summary = build_summary(dist_all)

print("Final summary built.")
print(final_summary.shape)
display(final_summary)


Building final summary
Saved summary: /home/f.capria/drug-discovery-lohi/results/results_fold_distance_tanimoto/hi/fold_distance_activity_detection_vs_random_bits_summary.csv (1107 rows)

Summary preview:
dataset model     pair  outer_fold   k              bit_source activity_protocol  pairwise_distance_mean  full_pairwise_distance_mean  delta_minus_full_pairwise  random_bits_pairwise_distance_mean  delta_minus_random_bits_pairwise  valid_molecule_fraction_mean  valid_pair_fraction_mean
   drd2    DT F1_vs_F2           1  10         activity_global            pooled                0.708013                     0.852592                  -0.144579                            0.942637                         -0.234625                      0.927044                  0.994881
   drd2    DT F1_vs_F2           1  10            activity_ood               ood                0.747359                     0.852592                  -0.105232                            0.942637                        

,dataset,dataset_label,model,pair,outer_fold,k,bit_source,activity_protocol,activity_fold,pairwise_distance_mean,pairwise_distance_std,valid_molecule_fraction_mean,valid_pair_fraction_mean,n_rows,full_pairwise_distance_mean,delta_minus_full_pairwise,random_bits_pairwise_distance_mean,delta_minus_random_bits_pairwise
0,drd2,DRD2,DT,F1_vs_F2,1,10,activity_global,pooled,NaN,0.708013,NaN,0.927044,0.994881,1,0.852592,-0.144579,0.942637,-0.234625
1,drd2,DRD2,DT,F1_vs_F2,1,10,activity_ood,ood,1.0,0.747359,NaN,0.906080,0.991433,1,0.852592,-0.105232,0.942637,-0.195278
2,drd2,DRD2,DT,F1_vs_F2,1,10,activity_random_shuffle,random,1.0,0.546577,NaN,0.989099,0.999882,1,0.852592,-0.306015,0.942637,-0.396060
3,drd2,DRD2,DT,F1_vs_F2,1,10,dataset_detection,same_search_cv,1.0,0.743797,NaN,0.949266,0.998169,1,0.852592,-0.108795,0.942637,-0.198841
4,drd2,DRD2,DT,F1_vs_F2,1,10,random_bits,NaN,NaN,0.942637,0.113057,0.231908,0.390877,30,0.852592,0.090046,0.942637,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1102,sol,Sol,SVM,F2_vs_F3,3,500,activity_ood,ood,3.0,0.902210,NaN,1.000000,1.000000,1,0.873216,0.028994,0.864865,0.037345
1103,sol,Sol,SVM,F2_vs_F3,3,500,activity_random_shuffle,random,3.0,0.902210,NaN,1.000000,1.000000,1,0.873216,0.028994,0.864865,0.037345
1104,sol,Sol,SVM,F2_vs_F3,3,500,dataset_detection,same_search_cv,3.0,0.915788,NaN,1.000000,1.000000,1,0.873216,0.042572,0.864865,0.050923
1105,sol,Sol,SVM,F2_vs_F3,3,500,random_bits,NaN,NaN,0.864865,0.036410,1.000000,1.000000,30,0.873216,-0.008351,0.864865,0.000000


In [17]:
print_section("Final checks")

print("Rows by dataset / model / bit_source:")
print(
    dist_all.groupby(["dataset", "model", "bit_source"])
    .size()
    .rename("n")
    .reset_index()
    .to_string(index=False)
)

print("\nUnique models:")
print(sorted(dist_all["model"].unique()))

print("\nUnique bit sources:")
print(sorted(dist_all["bit_source"].unique()))

print("\nK values:")
print(sorted(dist_all["k"].unique()))

print("\nDatasets:")
print(sorted(dist_all["dataset"].unique()))

unexpected_datasets = set(dist_all["dataset"].unique()) - set(DATASETS)
assert not unexpected_datasets, f"Unexpected datasets found: {unexpected_datasets}"

assert "kdr" not in set(
    dist_all["dataset"].astype(str).str.lower()
), "KDR leaked into the analysis."

observed_k = set(dist_all.loc[dist_all["bit_source"] != "full_ecfp4", "k"].unique())
expected_k = set(K_VALUES)
assert (
    observed_k == expected_k
), f"Unexpected k values. Observed={sorted(observed_k)}, expected={sorted(expected_k)}"

expected_bit_sources = {
    "full_ecfp4",
    "activity_global",
    "activity_ood",
    "activity_random_shuffle",
    "dataset_detection",
    "random_bits",
}

missing_sources = expected_bit_sources - set(dist_all["bit_source"].unique())
assert not missing_sources, f"Missing bit sources: {missing_sources}"

missing_models = set(MODELS) - set(dist_all["model"].unique())
assert not missing_models, f"Missing models: {missing_models}"

if "pairwise_distance" not in dist_all.columns:
    raise RuntimeError("Missing main metric column: pairwise_distance")

if dist_all["pairwise_distance"].notna().sum() == 0:
    raise RuntimeError("All pairwise_distance values are NaN — pipeline failed.")

print(
    f"\nFinite pairwise_distance: "
    f"{dist_all['pairwise_distance'].notna().sum()}/{len(dist_all)}"
)

if "valid_pair_fraction" in dist_all.columns:
    print("\nValid pair fraction summary:")
    print(
        dist_all["valid_pair_fraction"]
        .describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
        .to_string()
    )

if "valid_molecule_fraction" in dist_all.columns:
    print("\nValid molecule fraction summary:")
    print(
        dist_all["valid_molecule_fraction"]
        .describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
        .to_string()
    )

print("\nOK: model-wise and task-wise Tanimoto distance table completed.")


Final checks
Rows by dataset / model / bit_source:
dataset model              bit_source   n
   drd2    DT         activity_global  24
   drd2    DT            activity_ood  24
   drd2    DT activity_random_shuffle  24
   drd2    DT       dataset_detection  24
   drd2    DT              full_ecfp4   3
   drd2    DT             random_bits 720
   drd2    LR         activity_global  24
   drd2    LR            activity_ood  24
   drd2    LR activity_random_shuffle  24
   drd2    LR       dataset_detection  24
   drd2    LR              full_ecfp4   3
   drd2    LR             random_bits 720
   drd2   SVM         activity_global  24
   drd2   SVM            activity_ood  24
   drd2   SVM activity_random_shuffle  24
   drd2   SVM       dataset_detection  24
   drd2   SVM              full_ecfp4   3
   drd2   SVM             random_bits 720
    hiv    DT         activity_global  24
    hiv    DT            activity_ood  24
    hiv    DT activity_random_shuffle  24
    hiv    DT       data